<a href="https://colab.research.google.com/github/mvssilva/desafio-assistente-de-voz/blob/main/assistente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
language = 'pt'

Baseado no exemplo do desafio, propus adicionar uma etapa de ambiente aonde vai ser realizado uma busca em algum documento a partir da pergunta enviado por audio.

#1. Preparação Banco de Dados

In [1]:
!pip install -q langchain langchain-community langchain-text-splitters chromadb pypdf tiktoken
!pip install -q -U langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from google.colab import userdata

chave_secreta = userdata.get('KEY_API_GEMINI')
os.environ["GOOGLE_API_KEY"] = chave_secreta

# Carregando documento
loader = PyPDFLoader("/content/relatorio_final_marcos_silva_2025.pdf")
docs = loader.load()

# Fatir o texto em blocos menores
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap= 100)
splits = text_splitter.split_documents(docs)

# Criando banco de dados temporário na memoria
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview"),
    collection_name="nova_collection"
)


retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Artigo processado e pronto para consultas!")

Artigo processado e pronto para consultas!


# 2. Gravação de uma Pergunta por Áudio
Pontos de atenção para o funcionamento:
Esse código é exclusivo para o Google Colab, utiliza de biblioteca interna "google.colab" para comunicação entre o JavaScript do navegador e o backend em Python. Não funciona em Jupyter Notebooks padrões.

In [7]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 3. Célula do Assistente


In [12]:
!pip install -q git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 4.6 MB/s eta 0:00:00


In [15]:
import whisper
import google.generativeai as genai
import os

# configurando a API do gemini
genai.configure(api_key = os.environ["GOOGLE_API_KEY"])

# selecionando o modelo
modelo_gemini = genai.GenerativeModel('gemini-2.5-flash')

# captura e transcreve o que foi falado no audio
model_whisper = whisper.load_model("small")

result = model_whisper.transcribe(record_file, fp16=False, language = language)
pergunta = result["text"]

print(f"Sua pergunta foi: {pergunta}")

# busca no documento
docs_recuperados = retriever.invoke(pergunta)

contexto_recuperado = "\n\n".join([doc.page_content for doc in docs_recuperados])
print("\n--- Trechos recuperados do seu relatório ---")
print(contexto_recuperado[:300] + "...\n")

# chamada ao gemini com contexto
prompt = f"""Você é um assistente acadêmico especialista.
Responda à pergunta do usuário baseando-se EXCLUSIVAMENTE no contexto fornecido abaixo.
Se a informação solicitada não estiver no contexto, responda educadamente:
'Desculpe, não encontrei essa informação no documento fornecido.'

CONTEXTO DO DOCUMENTO:
{contexto_recuperado}

PERGUNTA DO USUÁRIO:
{pergunta}
"""

response = modelo_gemini.generate_content(prompt)
resposta_ia = response.text

print(f"Resposta da IA: {resposta_ia}")

Sua pergunta foi:  Qual o problema que é trabalhado neste documento?

--- Trechos recuperados do seu relatório ---
Universidade Federal do Espírito Santo
Programa Institucional de Iniciação Científica
Relatório Final de Pesquisa
Ciências Exatas e da Terra
4.1 O Problema d-MBV
A formulação do problema MBV inicia-se com um grafo G conexo, não direcionado e não ponderado, carac-
terizado por seu conjunto de vértice...

Resposta da IA: O problema trabalhado neste documento é o Problema d-MBV.


#4. Geração de Áudio de Resposta com gTTS



In [18]:
!pip install -q gTTS

In [19]:
from gtts import gTTS

# pegar a resposta da IA
texto_para_voz = resposta_ia

# Cria um objeto gTTS com a resposta gerada pelo ChatGPT e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=texto_para_voz, lang=language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))